# E1 — tuning A.1 (lr, weight decay por arquitetura)

Protocolo (seção 2.2, A.1), em duas etapas:

1. **Busca** (`configs/sweep/e1-tuning-search.yaml`): Optuna **TPE** (seed 42, 10 trials aleatórios de
   aquecimento, `multivariate`) com **MedianPruner** (a partir de 5 trials completos e do passo 1000).
   **25 trials por arquitetura**, um study por arquitetura, **4000 passos** por trial. Os 10 pontos de
   aquecimento são os mesmos em todas as arquiteturas (`src/tuning.py`).
2. **Confirmação** (`configs/sweep/e1-tuning-confirm.yaml`): as **3 melhores** configurações não podadas
   de cada arquitetura, retreinadas do zero por **8000 passos**, sem pruning. A melhor por arquitetura é
   a congelada.

Comum às duas: treino **só no fold 0** do desenvolvimento (16 pacientes); seleção pelo **Dice médio por
paciente** na validação fixa (11 pacientes); nenhum teste é carregado. AdamW + Cosine
(`T_max = max_steps`), **batch de 8** rotulados, `binary_all`, 256×256. Arquiteturas: U-Net, U-Net++,
FPN, DeepLabv3+ (ResNet34/ImageNet, smp) e SwinUNet tiny.

Resultado: `configs/frozen/e1-hparams.yaml`. **Fazer o commit desse arquivo é o congelamento**: depois
dele, nada de reajustar com folds avaliados ou com teste.

**Cuidado ao ler as tabelas.** Re-runs idênticos do X1 divergiram até ~0,10 de Dice. Uma diferença
entre trials menor que o ruído entre re-runs não é evidência de que um ponto é melhor — é por isso que a
etapa 2 retreina as 3 primeiras em vez de confiar na ordem da busca.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sys
sys.path.append("../../../")

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.sweep import Sweep, submit_sweep, sweep_status, watch_sweep
from src.tuning import rank_trials, write_frozen_hparams

SEARCH_SWEEP = "configs/sweep/e1-tuning-search.yaml"
CONFIRM_SWEEP = "configs/sweep/e1-tuning-confirm.yaml"
FROZEN_HPARAMS = "configs/frozen/e1-hparams.yaml"

search = Sweep(SEARCH_SWEEP)

/home/caioseda/miniconda3/envs/vfss/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## Os dados do tuning

Treino = fold 0 do desenvolvimento; validação = os 11 pacientes fixas (sem teste). A tabela sai dos mesmos
datasets que o sweep monta. O que importa ao ler: poucos frames por paciente (a validação é por paciente) e
uma época de ~15 passos, então 4000 passos são centenas de épocas sobre o mesmo punhado de frames.

In [4]:
# Estatísticas dos splits do tuning, montados como o sweep monta (só o fold 0 treina; validação fixa).
# Só constrói os datasets (tabela de frames) e lê as máscaras já redimensionadas; não treina nada.
import math

import numpy as np
from omegaconf import OmegaConf

from src.data.splits import resolve_project_path
from src.data.vfss_frame_dataset import VFSSIncaTrain, VFSSIncaVal

cfg = search.plan.config
base_data = OmegaConf.load(resolve_project_path(cfg.base_configs["unet"])).data
ds_params = OmegaConf.to_container(base_data.params.shared_dataset_params, resolve=True)
ds_params.update(split_file=str(cfg.data.split_file), train_folds=[int(f) for f in cfg.data.train_folds], cv_fold=None)
confirm_steps = int(Sweep(CONFIRM_SWEEP).plan.config.training.max_steps)
batch_size, drop_last = int(cfg.training.batch_size), bool(base_data.params.drop_last_train)

splits = {"treino": VFSSIncaTrain(**ds_params), "validação": VFSSIncaVal(**ds_params)}

rows = {}
for name, ds in splits.items():
    df = ds.video_frame_df
    fg = np.array([float((ds[i]["segmentation"] > 0).float().mean()) for i in range(len(ds))])
    per_patient = df.groupby("paciente_id").size()
    rows[name] = {
        "pacientes": df.paciente_id.nunique(),
        "vídeos": df.video_id.nunique(),
        "frames rotulados": len(df),
        "frames / paciente (mín–mediana–máx)": f"{per_patient.min()}–{per_patient.median():.0f}–{per_patient.max()}",
        "frames / vídeo (mediana)": df.groupby("video_id").size().median(),
        "rotuladores": ", ".join(f"{k}={v}" for k, v in df.selected_labeler.value_counts().items()),
        "classes (target)": f"{ds_params['target_variant']}: 0=fundo, 1=estrutura",
        "frames com máscara vazia": int((fg == 0).sum()),
        "% pixels de estrutura (média)": round(100 * fg.mean(), 2),
        "% pixels de estrutura (mín–máx)": f"{100 * fg.min():.2f}–{100 * fg.max():.2f}",
        "resolução": f"{ds_params['size']}×{ds_params['size']}",
    }
stats = pd.DataFrame(rows)

n_train = len(splits["treino"])
steps_epoch = n_train // batch_size if drop_last else math.ceil(n_train / batch_size)
stats.loc["batch (rotulados)"] = [batch_size, "—"]
stats.loc["passos por época (treino)"] = [steps_epoch, "—"]
stats.loc["frames descartados por época (drop_last)"] = [n_train - steps_epoch * batch_size if drop_last else 0, "—"]
stats.loc[f"épocas em {cfg.training.max_steps} passos (busca)"] = [round(cfg.training.max_steps / steps_epoch, 1), "—"]
stats.loc[f"épocas em {confirm_steps} passos (confirmação)"] = [round(confirm_steps / steps_epoch, 1), "—"]
stats.loc["validações por trial (busca)"] = [cfg.training.max_steps // cfg.training.val_every_n_steps, "—"]
stats.loc["pacientes em comum treino/validação"] = [
    len(set(splits["treino"].video_frame_df.paciente_id) & set(splits["validação"].video_frame_df.paciente_id)), "—"]
stats

,treino,validação
pacientes,16,11
vídeos,27,26
frames rotulados,121,110
frames / paciente (mín–mediana–máx),3–4–50,4–4–43
frames / vídeo (mediana),5.0,4.0
rotuladores,"VC=55, BB=22, VR=18, CS=18, AM=8","VC=48, BB=24, CS=24, VR=8, AM=6"
classes (target),"binary_all: 0=fundo, 1=estrutura","binary_all: 0=fundo, 1=estrutura"
frames com máscara vazia,0,0
% pixels de estrutura (média),2.19,2.68
% pixels de estrutura (mín–máx),1.35–3.19,1.30–4.81


## Etapa 1 — busca TPE (25 trials × 4000 passos)

In [5]:
# sweep_status(SEARCH_SWEEP)

In [ ]:
# Lança a busca destacada (tmux) e devolve na hora. Rode SÓ depois de commitar o split.
# submit_sweep(SEARCH_SWEEP)

SyntaxError: unmatched ')' (2872346888.py, line 2)

In [ ]:
# Acompanha. Interromper esta célula para de OLHAR, não de treinar.
watch_sweep(SEARCH_SWEEP)

e1-tuning-search: 81/125 celulas concluidas (12:44:42)
               run_id  seed  trial          arch              status
         unet-t00-s42    42      0          unet       done (na CSV)
 unetplusplus-t00-s42    42      0  unetplusplus       done (na CSV)
          fpn-t00-s42    42      0           fpn       done (na CSV)
deeplabv3plus-t00-s42    42      0 deeplabv3plus       done (na CSV)
swinunet_tiny-t00-s42    42      0 swinunet_tiny       done (na CSV)
         unet-t01-s42    42      1          unet       done (na CSV)
 unetplusplus-t01-s42    42      1  unetplusplus       done (na CSV)
          fpn-t01-s42    42      1           fpn       done (na CSV)
deeplabv3plus-t01-s42    42      1 deeplabv3plus       done (na CSV)
swinunet_tiny-t01-s42    42      1 swinunet_tiny       done (na CSV)
         unet-t02-s42    42      2          unet       done (na CSV)
 unetplusplus-t02-s42    42      2  unetplusplus       done (na CSV)
          fpn-t02-s42    42      2           fpn

### Histórico dos studies

Um study por arquitetura em `logs/e1-tuning-search/_sweep/optuna.journal`. `value` é o Dice médio por
paciente na validação (vazio nos podados).

In [ ]:
trials = search.plan.trials_dataframe()
print(trials.groupby(["archz", "state"]).size().unstack(fill_value=0))
trials[["arch", "number", "state", "value", "params_lr", "params_weight_decay"]].round(4)

In [ ]:
# Onde o TPE concentrou a busca. x = podado; a cor é o Dice dos completos.
archs = trials.arch.unique()
fig, axes = plt.subplots(1, len(archs), figsize=(4 * len(archs), 3.5), sharex=True, sharey=True, squeeze=False)
for ax, arch in zip(axes[0], archs):
    t = trials[trials.arch == arch]
    done, pruned = t[t.state == "COMPLETE"], t[t.state == "PRUNED"]
    points = ax.scatter(done.params_lr, done.params_weight_decay, c=done.value, cmap="viridis")
    ax.scatter(pruned.params_lr, pruned.params_weight_decay, marker="x", color="gray")
    ax.set(xscale="log", yscale="log", title=arch, xlabel="lr")
    fig.colorbar(points, ax=ax)
axes[0][0].set_ylabel("weight_decay")
fig.tight_layout()

### Ranking da busca (só trials completos) e candidatos da etapa 2

In [ ]:
search_results = search.load_results()
assert search_results.empty or set(search_results.eval_set) == {"val"}, "o tuning so pode ter linhas de validacao"

ranked_search = rank_trials(search_results, by=["arch"])
ranked_search[ranked_search["rank"] <= 3][["arch", "rank", "trial", "lr", "weight_decay", "dice"]]

## Etapa 2 — confirmação: top-3 × 8000 passos

Só lista e treina com a busca completa para todas as arquiteturas.

In [ ]:
sweep_status(CONFIRM_SWEEP)

In [ ]:
# submit_sweep(CONFIRM_SWEEP)

watch_sweep(CONFIRM_SWEEP)

## Resultados da confirmação: Dice médio por paciente na validação fixa

In [ ]:
confirm = Sweep(CONFIRM_SWEEP)
results = confirm.load_results()
assert results.empty or set(results.eval_set) == {"val"}, "o tuning so pode ter linhas de validacao"

ranked = rank_trials(results, by=["arch"]).merge(
    ranked_search[["arch", "trial", "rank", "dice"]].rename(columns={"rank": "rank_busca", "dice": "dice_busca"}),
    on=["arch", "trial"],
)
ranked[["arch", "rank", "trial", "lr", "weight_decay", "dice", "rank_busca", "dice_busca"]].round(4)

In [ ]:
best = ranked[ranked["rank"] == 1]
best[["arch", "trial", "lr", "weight_decay", "dice"]]

## Convergência: 8000 passos bastam?

Curva de `val/dice_patient` da configuração escolhida de cada arquitetura. Se ela ainda estiver subindo
no fim, o protocolo pede para reavaliar `max_steps` (e `T_max` junto) **antes** de congelar.

In [ ]:
run_dirs = (results.drop_duplicates("run_id").set_index(["arch", "trial"]).output_dir)

fig, ax = plt.subplots(figsize=(8, 4))
for _, row in best.iterrows():
    metrics = pd.read_csv(Path(run_dirs[(row.arch, row.trial)]) / "csv" / "metrics.csv")
    curve = metrics.dropna(subset=["val/dice_patient"])
    ax.plot(curve.step, curve["val/dice_patient"], marker="o", label=f"{row.arch} (t{row.trial})")
ax.set_xlabel("passo"); ax.set_ylabel("val/dice_patient"); ax.legend(); ax.grid(alpha=0.3)

## Congelamento

Grava `configs/frozen/e1-hparams.yaml` com a melhor configuração da **confirmação** por arquitetura
(recusa sobrescrever). Commite o arquivo em seguida.

In [ ]:
# write_frozen_hparams(best, FROZEN_HPARAMS,
#                      source=f"{confirm.results_csv} (busca: {search.results_csv})")